In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Representa una pieza con atributos simulados
class Piece:
    def __init__(self, texture, symmetry, edges, center_offset, label):
        self.texture = texture
        self.symmetry = symmetry
        self.edges = edges
        self.center_offset = center_offset
        self.label = label  # "Correcta" o "Defectuosa"

    def to_vector(self):
        return [self.texture, self.symmetry, self.edges, self.center_offset]

# Genera piezas simuladas con reglas básicas para determinar si están defectuosas
class PieceDatasetGenerator:
    def __init__(self, n=400):
        self.n = n

    def generate(self):
        pieces = []
        np.random.seed(42)
        
        for _ in range(self.n):
            # Generar características según distribuciones normales
            texture = np.clip(np.random.normal(0.5, 0.15), 0, 1)
            symmetry = np.clip(np.random.normal(0.6, 0.2), 0, 1)
            edges = max(0, np.random.normal(50, 15))
            center_offset = max(0, np.random.normal(0.2, 0.1))
            
            # Aplicar reglas de clasificación
            if ((symmetry < 0.4 and center_offset > 0.25) or 
                texture < 0.35 or 
                edges < 30 or 
                center_offset > 0.35):
                label = "Defectuosa"
            else:
                label = "Correcta"
            
            pieces.append(Piece(texture, symmetry, edges, center_offset, label))
        
        return pieces

# Modelo SVM para clasificar piezas
class PieceClassifier:
    def __init__(self):
        self.model = SVC(kernel='rbf', gamma='scale', C=1.0)

    def fit(self, pieces):
        X = [piece.to_vector() for piece in pieces]
        y = [piece.label for piece in pieces]
        self.model.fit(X, y)

    def predict(self, texture, symmetry, edges, offset):
        X = [[texture, symmetry, edges, offset]]
        return self.model.predict(X)[0]

    def evaluate(self, test_data):
        X_test = [piece.to_vector() for piece in test_data]
        y_test = [piece.label for piece in test_data]
        y_pred = self.model.predict(X_test)
        
        print("📊 Matriz de confusión:")
        print(confusion_matrix(y_test, y_pred))
        print("\n📝 Informe de clasificación:")
        print(classification_report(y_test, y_pred))

# Ejemplo de uso y visualización
class PieceAnalysisExample:
    def run(self):
        # 1. Generación de datos
        generator = PieceDatasetGenerator()
        pieces = generator.generate()
        
        # 2. División de datos
        train_data, test_data = train_test_split(pieces, test_size=0.3, random_state=42)
        
        # 3. Entrenamiento
        classifier = PieceClassifier()
        classifier.fit(train_data)
        
        # 4. Evaluación
        classifier.evaluate(test_data)
        
        # 5. Predicción personalizada
        print("\n🔎 Predicción de pieza personalizada:")
        texture, symmetry, edges, offset = 0.45, 0.5, 45, 0.15
    
 # Ejecutar ejemplo
example = PieceAnalysisExample()
example.run()

📊 Matriz de confusión:
[[90  0]
 [23  7]]

📝 Informe de clasificación:
              precision    recall  f1-score   support

    Correcta       0.80      1.00      0.89        90
  Defectuosa       1.00      0.23      0.38        30

    accuracy                           0.81       120
   macro avg       0.90      0.62      0.63       120
weighted avg       0.85      0.81      0.76       120


🔎 Predicción de pieza personalizada:
